In [10]:
import os
import sys
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split

from copy import deepcopy
from string import ascii_lowercase

parent = os.path.abspath("..")
if parent not in sys.path:
    sys.path.append(parent)

from src.config import settings
from src.modeling import translation_model


In [4]:
net = deepcopy(translation_model.net)
transform = deepcopy(translation_model.transform)
BATCH_SIZE = settings.BATCH_SIZE
WORKERS = settings.WORKERS
RATIO_VALIDATION = settings.RATIO_VALIDATION

In [5]:
save_path = "../models/runs/translation/train4/best_model_epoch92.pth"

checkpoint = torch.load(save_path, weights_only=True)
net.load_state_dict(checkpoint["model_state_dict"])
epoch = checkpoint["epoch"]
loss = checkpoint["loss"]

text_accuracy = (
    f" y exactitud {checkpoint['accuracy']:.4%}" if "accuracy" in checkpoint else ""
)

print(f"Cargado modelo de la época {epoch}, con pérdida {loss:.4f}{text_accuracy} --> Entrenamiento")

Cargado modelo de la época 92, con pérdida 2.3221 y exactitud 100.0000% --> Entrenamiento


## Validación datos de testing

In [6]:
custom_dataset = translation_model.CustomDataset(root="../data/processed/kaggle/Braille Dataset", transform=transform)
dataloader = DataLoader(
    custom_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS
)

# Split dataset into training and validation sets
train_size = int(len(custom_dataset) * (1 - RATIO_VALIDATION))
validation_size = len(custom_dataset) - train_size

dataset_train, dataset_val = random_split(
    custom_dataset,
    [train_size, validation_size],
    generator=torch.Generator().manual_seed(42),
)

# Create data loaders
val_loader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)

y_predicted = []
y_true = []

with torch.no_grad():
    for images, labels_true in val_loader:
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)

        images = images.numpy().reshape(
            images.shape[0], images.shape[2], images.shape[3]
        )  # (batch_size, height, width)
        for i in range(images.shape[0]):
            img = images[i, :, :]

            img_path = custom_dataset.imgs_path[i]
            character_predicted = ascii_lowercase[predicted[i]]
            character_true = ascii_lowercase[labels_true[i]]
            y_predicted.append(character_predicted)
            y_true.append(character_true)

In [7]:
accuracy = accuracy_score(y_true, y_predicted)
print(f"Accuracy: {accuracy:.4%}")

Accuracy: 98.5470%


In [14]:
f1  = f1_score(y_true, y_predicted, average="weighted")
print(f"F1 Score: {f1:.4%}")

F1 Score: 98.5762%


In [66]:
def plot_confusion_matrix(
    y_true: list[str],
    y_predicted: list[str],
    normalize: str | None = None,
    fmt: str = "",
) -> None:
    labels = [ch for ch in ascii_lowercase]
    fig, ax = plt.subplots(figsize=(10, 10))

    cm = confusion_matrix(y_true, y_predicted, normalize=normalize)

    title = f" Normalized {normalize.capitalize()}" if normalize else ""

    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax, pad=0.01, shrink=0.845)
    ax.set(
        xticks=range(n := len(labels)),
        yticks=range(n),
        xticklabels=labels,
        yticklabels=labels,
        ylabel="True",
        xlabel="Predicted",
        title=f"Confusion Matrix{title}",
    )

    plt.setp(ax.get_xticklabels(), rotation=15, size=10)
    plt.setp(ax.get_yticklabels(), rotation=25, size=10)

    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                format(cm[i, j], fmt),
                ha="center",
                va="center",
                size=6,
                color="white" if cm[i, j] > thresh else "black",
            )

    plt.show()

    normalize_label = f"_normalized_{normalize}" if normalize else ""
    fig.savefig(f"../models/runs/translation/train4/confustion_matrix{normalize_label}.png")

In [69]:
%%capture
plot_confusion_matrix(y_true, y_predicted, None, "")
plot_confusion_matrix(y_true, y_predicted, "pred", ".2f")
plot_confusion_matrix(y_true, y_predicted, "true", ".2f")
plot_confusion_matrix(y_true, y_predicted, "all", ".2f")

## Validación con imagenes reales (en [data/processed/character/](../data/processed/character/))

In [13]:
class CustomDataset(Dataset):
    ALLOWED_EXTENSIONS: list[str] = settings.VALID_IMAGES_EXTENSIONS

    def __init__(self, root: str, transform: transforms.Compose | None = None) -> None:
        if not os.path.exists(root):
            raise FileNotFoundError(f"Path {root} does not exist")

        self.root: Path = Path(root)
        self.transform = transform

        images_path = [
            list(self.root.rglob(f"*{ext}")) for ext in self.ALLOWED_EXTENSIONS
        ]

        self.imgs_path: list[Path] = [
            item for sublist in images_path for item in sublist
        ]

    def __len__(self) -> int:
        return len(self.imgs)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        img_path = self.imgs_path[index]

        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        img = img.astype(np.float32) / 255.0  # Normaliza a [0, 1]

        if self.transform:
            img = self.transform(img)
        else:
            img = torch.from_numpy(img).float()

        # Ignorar el segundo elemento de la tupla
        label = torch.from_numpy(np.uint16([index]))

        # if ngpus > 0:
        #     img = torch.cuda.FloatTensor(img)
        #     label = label.to(device)

        return img, label

    @property
    def imgs(self) -> list[Path]:
        return self.imgs_path

In [8]:
custom_dataset = CustomDataset(
    root="../data/processed/character",
    transform=transform,
)

dataloader = DataLoader(
    custom_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS
)

In [7]:
rows, columns = 20, 257
fig, axes = plt.subplots(rows, columns, figsize=(70, 7))
axes_flatten = axes.flatten()

cont = 0
with torch.no_grad():
    for images, _ in dataloader:
        outputs = net(images)
        labels, predicted = torch.max(outputs, 1)

        images = images.numpy().reshape(
            images.shape[0], images.shape[2], images.shape[3]
        )  # (batch_size, height, width)
        labels = labels.numpy()
        for i in range(images.shape[0]):
            img = images[i, :, :]
            img_path = custom_dataset.imgs_path[i]

            ax = axes_flatten[cont]

            character_predicted = ascii_lowercase[predicted[i]]
            ax.set(xticks=[], yticks=[])
            ax.set_xlabel(img_path.name, fontsize=2, labelpad=1)
            ax.set_title(character_predicted, fontsize=3, pad=1)

            scale = 10
            img = cv2.resize(
                img, (img.shape[1] * scale, img.shape[0] * scale), interpolation=cv2.INTER_CUBIC
            )
            ax.imshow(img, cmap="gray")
            cont += 1

fig.savefig("../models/runs/translation/train4/predictions.svg", format="svg", dpi=500)
plt.close()